In [4]:
import numpy as np, pandas as pd
songs = pd.read_csv('./week3/Data_processing/recommend/korean_music.csv')
songs

,song_id,title,genre,mood,tempo,vocal,instrument,year
0,song1,봄날,Ballad,"romantic,sad",slow,male solo,piano,2010s
1,song2,Dynamite,Pop,"energetic,happy",fast,group,synth,2020s
2,song3,Love Scenario,Hip-hop,"romantic,chill",medium,male solo,piano,2010s
3,song4,좋은 날,Pop,"happy,energetic",fast,female solo,piano,2010s
4,song5,Cherry Blossom Ending,Ballad,"romantic,chill",slow,male solo,"guitar,piano",2010s
5,song6,Fantastic Baby,K-pop,"energetic,dark",fast,group,synth,2010s
6,song7,밤편지,Ballad,"romantic,sad",slow,female solo,piano,2010s
7,song8,어떻게 이별까지 사랑하겠어,Ballad,"sad,romantic",slow,male solo,piano,2010s
8,song9,강남스타일,K-pop,"energetic,happy",fast,male solo,synth,2010s
9,song10,사랑했지만,Ballad,sad,slow,male solo,guitar,1990s


In [7]:
#make list by comma
songs['mood'] = songs['mood'].fillna('').apply(lambda x: x.split(','))
songs['instrument'] = songs['instrument'].fillna('').apply(lambda x: x.split(','))
songs['idx'] = songs.index
songs

,song_id,title,genre,mood,tempo,vocal,instrument,year,idx
0,song1,봄날,Ballad,"[romantic, sad]",slow,male solo,[piano],2010s,0
1,song2,Dynamite,Pop,"[energetic, happy]",fast,group,[synth],2020s,1
2,song3,Love Scenario,Hip-hop,"[romantic, chill]",medium,male solo,[piano],2010s,2
3,song4,좋은 날,Pop,"[happy, energetic]",fast,female solo,[piano],2010s,3
4,song5,Cherry Blossom Ending,Ballad,"[romantic, chill]",slow,male solo,"[guitar, piano]",2010s,4
5,song6,Fantastic Baby,K-pop,"[energetic, dark]",fast,group,[synth],2010s,5
6,song7,밤편지,Ballad,"[romantic, sad]",slow,female solo,[piano],2010s,6
7,song8,어떻게 이별까지 사랑하겠어,Ballad,"[sad, romantic]",slow,male solo,[piano],2010s,7
8,song9,강남스타일,K-pop,"[energetic, happy]",fast,male solo,[synth],2010s,8
9,song10,사랑했지만,Ballad,[sad],slow,male solo,[guitar],1990s,9


In [9]:
# multi value => one hot encoding -> by song
mood_oh = pd.get_dummies(
    songs.explode('mood')[['idx', 'mood']],columns=['mood']
).groupby('idx').max()
inst_oh = pd.get_dummies(
    songs.explode('instrument')[['idx', 'instrument']], columns=['instrument']
).groupby('idx').max()
print(mood_oh)
# sole value -> one hot encoded
single_oh = pd.get_dummies(songs[['genre', 'tempo', 'vocal', 'year']])
#combine, join data
features = pd.concat([single_oh, mood_oh, inst_oh], axis =1).astype(int)
features.index = songs['title']
print('number of songs, number of features', features.shape)

     mood_chill  mood_dark  mood_energetic  mood_happy  mood_romantic  \
idx                                                                     
0         False      False           False       False           True   
1         False      False            True        True          False   
2          True      False           False       False           True   
3         False      False            True        True          False   
4          True      False           False       False           True   
5         False       True            True       False          False   
6         False      False           False       False           True   
7         False      False           False       False           True   
8         False      False            True        True          False   
9         False      False           False       False          False   
10        False      False           False       False           True   
11        False      False            True        T

In [11]:
from sklearn.metrics.pairwise import cosine_similarity
sim = cosine_similarity(features.values)
sim_df = pd.DataFrame(sim, index = features.index, columns=features.index)
sim_df

title,봄날,Dynamite,Love Scenario,좋은 날,Cherry Blossom Ending,Fantastic Baby,밤편지,어떻게 이별까지 사랑하겠어,강남스타일,사랑했지만,...,서시,한 페이지가 될 수 있게,나는 나비,오래된 노래,사건의 지평선,Hype Boy,Ditto,좋아좋아,거짓말,사랑의 배터리
title,,,,,,,,,,,,,,,,,,,,,
봄날,1.000000,0.000000,0.571429,0.285714,0.801784,0.142857,0.857143,1.000000,0.285714,0.617213,...,0.771517,0.142857,0.000000,0.857143,0.285714,0.000000,0.285714,0.571429,0.142857,0.000000
Dynamite,0.000000,1.000000,0.000000,0.571429,0.000000,0.571429,0.000000,0.000000,0.571429,0.000000,...,0.000000,0.428571,0.428571,0.000000,0.285714,0.428571,0.285714,0.142857,0.142857,0.428571
Love Scenario,0.571429,0.000000,1.000000,0.285714,0.668153,0.142857,0.428571,0.571429,0.285714,0.154303,...,0.308607,0.142857,0.000000,0.428571,0.571429,0.285714,0.285714,0.285714,0.285714,0.000000
좋은 날,0.285714,0.571429,0.285714,1.000000,0.267261,0.428571,0.428571,0.285714,0.571429,0.000000,...,0.154303,0.571429,0.428571,0.142857,0.428571,0.142857,0.000000,0.428571,0.000000,0.571429
Cherry Blossom Ending,0.801784,0.000000,0.668153,0.267261,1.000000,0.133631,0.668153,0.801784,0.267261,0.577350,...,0.577350,0.267261,0.133631,0.801784,0.400892,0.133631,0.400892,0.534522,0.000000,0.000000
Fantastic Baby,0.142857,0.571429,0.142857,0.428571,0.133631,1.000000,0.142857,0.142857,0.714286,0.000000,...,0.000000,0.428571,0.285714,0.142857,0.000000,0.285714,0.285714,0.000000,0.285714,0.285714
밤편지,0.857143,0.000000,0.428571,0.428571,0.668153,0.142857,1.000000,0.857143,0.142857,0.462910,...,0.617213,0.142857,0.000000,0.714286,0.428571,0.000000,0.285714,0.714286,0.142857,0.142857
어떻게 이별까지 사랑하겠어,1.000000,0.000000,0.571429,0.285714,0.801784,0.142857,0.857143,1.000000,0.285714,0.617213,...,0.771517,0.142857,0.000000,0.857143,0.285714,0.000000,0.285714,0.571429,0.142857,0.000000
강남스타일,0.285714,0.571429,0.285714,0.571429,0.267261,0.714286,0.142857,0.285714,1.000000,0.154303,...,0.154303,0.571429,0.428571,0.285714,0.000000,0.428571,0.285714,0.142857,0.142857,0.428571


In [12]:
def recommend(title, n=5):
    scores = sim_df[title].drop(title).sort_values(ascending=False)
    return scores.head(n)
print('recommend similar songs of 밤편지')
print(recommend('밤편지').round(2))

recommend similar songs of 밤편지
title
봄날                0.86
어떻게 이별까지 사랑하겠어    0.86
좋니                0.77
좋아좋아              0.71
취중진담              0.71
Name: 밤편지, dtype: float64


In [25]:
#songs customer likes
liked =['봄날', '밤편지', '사건의 지평선']
profile = features.loc[liked].mean().values.reshape(1, -1)
profile


array([[0.66666667, 0.        , 0.        , 0.33333333, 0.        ,
        0.        , 0.        , 0.33333333, 0.66666667, 0.        ,
        0.        , 0.66666667, 0.        , 0.        , 0.33333333,
        0.        , 0.        , 0.66666667, 0.33333333, 0.33333333,
        0.        , 0.        , 0.        , 1.        , 0.66666667,
        0.        , 0.        , 1.        , 0.        ]])

In [26]:
scores = cosine_similarity(profile, features.values)[0]
result = pd.Series(scores, index=features.index).drop(liked).sort_values(ascending=False)
print('liked song:', liked)
print('\nrecommend:', result.head(5).round(2))

liked song: ['봄날', '밤편지', '사건의 지평선']

recommend: title
어떻게 이별까지 사랑하겠어           0.86
Cherry Blossom Ending    0.75
취중진담                     0.75
눈의 꽃                     0.75
좋니                       0.75
dtype: float64
